# 🤖 AI Engineering Fundamentals — Lezione 5
## Notebook Gruppo B

**ITS Novitas 4.0 | Giovedì 04/06/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "B"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
!pip install anthropic requests -q
from google.colab import userdata
import anthropic, os, requests

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

# Tool loop base — usatelo per tutti gli esercizi
def tool_loop(messaggio, tools, esegui_fn, system=None):
    """Loop agente generico con contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            return "Errore: loop non terminato"
        params = {"model": "claude-haiku-4-5-20251001",
                  "max_tokens": 1024, "tools": tools, "messages": history}
        if system:
            params["system"] = system
        response = client.messages.create(**params)
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            results = []
            for b in response.content:
                if b.type == "tool_use":
                    print(f"  🔧 {b.name}({b.input})")
                    r = esegui_fn(b.name, b.input)
                    print(f"  ✅ {str(r)[:100]}")
                    results.append({"type": "tool_result",
                                   "tool_use_id": b.id, "content": str(r)})
            history.append({"role": "user", "content": results})

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo B: Costruire Tool Personalizzati

Costruite 3 tool reali per il chatbot WiData:
calcolatrice sicura, meteo in tempo reale, Wikipedia.
Poi integrateli in un chatbot multi-tool.

---
### Esercizio 1 — Tool calcolatrice con validazione *(guidato)*

Costruite una calcolatrice sicura che usa `eval()` ma valida
l'input prima di eseguirlo. Dimostrate che blocca input malevoli.

In [ ]:
# Esercizio 1 — calcolatrice sicura

tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica precisa. Usa per calcoli aritmetici, percentuali, potenze.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "Espressione matematica. Es: '234 * 567', '15/100 * 847'"
            }
        },
        "required": ["espressione"]
    }
}

def calcola(espressione: str) -> str:
    # TODO: implementate la validazione con lista bianca
    # Caratteri consentiti: cifre, operatori, parentesi, spazi, punto
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in espressione):
        return ___  # messaggio di errore
    try:
        risultato = eval(espressione)
        return f"{espressione} = {risultato}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"

# Test sicurezza
test_input = [
    "234 * 567",                    # ✅ valido
    "15 / 100 * 847",               # ✅ valido
    "__import__('os').system('ls')", # ❌ da bloccare
    "open('/etc/passwd').read()",    # ❌ da bloccare
]

print("Test sicurezza calcolatrice:")
for inp in test_input:
    risultato = calcola(inp)
    stato = "✅" if "Errore" in risultato or "non valida" in risultato or "=" in risultato else "⚠️"
    print(f"{stato} Input: '{inp[:40]}' → {risultato}")

# Test con il chatbot
def esegui(nome, params):
    if nome == "calcola": return calcola(params["espressione"])
    return "Tool non trovato"

print()
print(tool_loop("Quanto fa il 22% di 1500?", [tool_calcolatrice], esegui))

---
### Esercizio 2 — Tool meteo con API reale *(guidato)*

Implementate il tool meteo usando open-meteo.com.
Gratuito, senza API key, funziona subito.

In [ ]:
# Esercizio 2 — tool meteo

CITTA_COORD = {
    "sassari":  {"lat": 40.7259, "lon": 8.5563},
    "cagliari": {"lat": 39.2238, "lon": 9.1217},
    "nuoro":    {"lat": 40.3207, "lon": 9.3311},
    "olbia":    {"lat": 40.9237, "lon": 9.4992},
    "roma":     {"lat": 41.9028, "lon": 12.4964},
    "milano":   {"lat": 45.4642, "lon": 9.1900},
}

tool_meteo = {
    "name": "get_meteo",
    "description": "Ottieni il meteo attuale per una città italiana. Usa quando l'utente chiede del tempo atmosferico.",
    "input_schema": {
        "type": "object",
        "properties": {
            "citta": {
                "type": "string",
                "description": "Nome città in minuscolo. Es: 'sassari', 'cagliari', 'roma'"
            }
        },
        "required": ["citta"]
    }
}

def get_meteo(citta: str) -> str:
    citta = citta.lower().strip()
    if citta not in CITTA_COORD:
        return f"Città '{citta}' non supportata. Disponibili: {', '.join(CITTA_COORD.keys())}"

    coord = CITTA_COORD[citta]
    # TODO: chiamate l'API open-meteo.com
    # URL: https://api.open-meteo.com/v1/forecast
    # Parametri: latitude, longitude, current=temperature_2m,weathercode,windspeed_10m
    # timezone=Europe/Rome
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={coord['lat']}&longitude={coord['lon']}"
        f"&current=temperature_2m,windspeed_10m,relative_humidity_2m"
        f"&timezone=Europe/Rome"
    )
    try:
        # TODO: fate la chiamata con timeout=5
        data = ___
        curr = data["current"]
        return (
            f"Meteo a {citta.title()}: {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, "
            f"vento {curr['windspeed_10m']} km/h."
        )
    except Exception as e:
        return f"Errore API meteo: {str(e)}"

# Test diretto
print(get_meteo("sassari"))
print(get_meteo("parigi"))  # città non in lista

# Test con chatbot
def esegui_v2(nome, params):
    if nome == "calcola":  return calcola(params["espressione"])
    if nome == "get_meteo": return get_meteo(params["citta"])
    return "Tool non trovato"

print()
print(tool_loop("Che tempo fa a Sassari e Cagliari?",
                [tool_calcolatrice, tool_meteo], esegui_v2))

---
### Esercizio 3 — Tool Wikipedia *(libero)*

Costruite il tool Wikipedia con fallback search.
Se la pagina non esiste, cercate con l'API di ricerca
e restituite il primo risultato.

In [ ]:
# Esercizio 3 — tool Wikipedia con fallback

tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "",  # ← scrivete voi la description
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Termine da cercare"},
            "lingua": {"type": "string", "description": "'it' o 'en'",
                      "enum": ["it", "en"]}
        },
        "required": ["query"]
    }
}

def cerca_wikipedia(query: str, lingua: str = "it") -> str:
    try:
        # Tentativo diretto
        url = f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5)

        if r.status_code == 404:
            # TODO: fallback — usate l'API di ricerca Wikipedia
            # https://{lingua}.wikipedia.org/w/api.php
            # action=opensearch, search=query, limit=1
            ___

        data = r.json()
        extract = data.get("extract", "Nessuna descrizione")
        return f"{data.get('title', query)}: {extract[:500]}"

    except requests.Timeout:
        return "Errore: Wikipedia non risponde (timeout)"
    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

# Test
print(cerca_wikipedia("Sassari"))
print(cerca_wikipedia("Anthropic", "en"))
print(cerca_wikipedia("termine che non esiste xyzxyz"))  # test fallback

# Aggiornate il router e testate con il chatbot
def esegui_v3(nome, params):
    if nome == "calcola":          return calcola(params["espressione"])
    if nome == "get_meteo":        return get_meteo(params["citta"])
    if nome == "cerca_wikipedia":  return cerca_wikipedia(params["query"], params.get("lingua", "it"))
    return "Tool non trovato"

TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia]

print()
print(tool_loop("Chi ha fondato Anthropic e in che anno?",
                TUTTI_I_TOOL, esegui_v3))

---
### Esercizio 4 — Chatbot completo con 3 tool *(libero)*

Integrate i 3 tool nel chatbot RAG della Lezione 4.
Il chatbot deve usare sia il contesto RAG che i tool
quando appropriato.

In [ ]:
# Esercizio 4 — chatbot completo RAG + 3 tool

# TODO: integrate chromadb e il documento WiData dalla Lezione 4
# Il chatbot deve:
# 1. Recuperare chunk RAG rilevanti per la domanda
# 2. Passarli come contesto nel system prompt
# 3. Avere a disposizione i 3 tool
# 4. Rispondere usando RAG per domande sui prodotti
#    e tool per calcoli, meteo, Wikipedia

SYSTEM_COMPLETO = """
Sei l'assistente di WiData Srl.
Per domande sui prodotti WiData usa il contesto fornito.
Per calcoli usa la calcolatrice.
Per il meteo usa il tool meteo.
Per informazioni generali usa Wikipedia.
Non inventare mai dati sui prodotti WiData.
"""

# Test con domande miste
domande_miste = [
    "Quanto costa il piano Pro di Xplore per 3 anni?",     # RAG + calcolatrice
    "Fa più caldo a Sassari o Cagliari oggi?",              # meteo 2 città
    "Chi ha inventato LoRaWAN, la tecnologia usata da XS200?",  # Wikipedia + RAG
]

# ...

---
## 📊 Preparate la presentazione (5 slide)

1. **La struttura di un tool** — i 3 campi con il vostro esempio migliore
2. **Validazione dell'input** — il test di sicurezza della calcolatrice
3. **Tool meteo in azione** — demo live con temperatura reale
4. **Tool concatenati** — domanda che usa RAG + calcolatrice insieme
5. **La vostra guida pratica** — quando usare RAG vs tool per rispondere

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*